# ML Experimentation — Surge Price Prediction
Trains the XGBoost surge multiplier model, logs metrics to MLflow, and compares
feature importance across experiments. This mirrors what `ml_retrain_dag.py`
runs on a schedule — the notebook is the interactive version for exploration.

In [ ]:
import sys
sys.path.insert(0, '..')

import mlflow
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split

from simulator.ride_generator import generate_ride_event
from ml.feature_engineering import build_features
from ml.train_surge_model import train_surge_model
from ml.evaluate_model import evaluate_regression

mlflow.set_tracking_uri('../data/mlflow')
print('MLflow tracking URI:', mlflow.get_tracking_uri())

## 1. Generate Synthetic Training Data

In [ ]:
N = 5000
rows = [generate_ride_event() for _ in range(N)]
df = pd.DataFrame(rows)
print(df.shape)
df.head()

In [ ]:
# Surge distribution
df['surge_multiplier'].hist(bins=30, figsize=(10, 4), color='steelblue', edgecolor='white')
plt.title('Surge Multiplier Distribution')
plt.xlabel('Surge Multiplier')
plt.ylabel('Count')
plt.tight_layout()
plt.show()

df['surge_multiplier'].describe()

## 2. Feature Engineering

In [ ]:
features, target = build_features(df)
print('Feature matrix shape:', features.shape)
print('Feature columns:', list(features.columns))

In [ ]:
# Correlation of numeric features with target
import numpy as np

numeric_features = ['distance_km', 'fare_base_inr', 'event_hour']
corr = pd.concat([features[numeric_features], target], axis=1).corr()[target.name]
corr.drop(target.name).sort_values().plot(kind='barh', figsize=(8, 4), title='Feature Correlation with Surge Multiplier')
plt.tight_layout()
plt.show()

## 3. Train XGBoost Model (logged to MLflow)

In [ ]:
model, metrics = train_surge_model(df, experiment_name='surge_price_prediction_notebook')
print('Metrics:', metrics)

## 4. Feature Importance

In [ ]:
importance = pd.Series(
    model.get_booster().get_score(importance_type='gain'),
).sort_values(ascending=True)

importance.tail(15).plot(
    kind='barh', figsize=(10, 6),
    title='Top 15 Features by Gain — XGBoost Surge Model',
    color='steelblue'
)
plt.tight_layout()
plt.show()

## 5. Residual Analysis

In [ ]:
x_train, x_test, y_train, y_test = train_test_split(features, target, test_size=0.2, random_state=42)
y_pred = model.predict(x_test)

residuals = y_test.values - y_pred

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.scatter(y_pred, residuals, alpha=0.4, color='steelblue')
ax1.axhline(0, color='red', linestyle='--')
ax1.set_title('Residuals vs Predicted')
ax1.set_xlabel('Predicted Surge')
ax1.set_ylabel('Residual')

ax2.scatter(y_test, y_pred, alpha=0.4, color='steelblue')
lims = [1.0, 3.5]
ax2.plot(lims, lims, 'r--')
ax2.set_title('Actual vs Predicted Surge')
ax2.set_xlabel('Actual')
ax2.set_ylabel('Predicted')

plt.tight_layout()
plt.show()

print('MAE:', round(metrics['mae'], 4))
print('RMSE:', round(metrics['rmse'], 4))
print('R²:', round(metrics['r2'], 4))

## 6. View MLflow Experiment Runs

In [ ]:
runs = mlflow.search_runs(experiment_names=['surge_price_prediction_notebook'])
runs[['start_time', 'metrics.mae', 'metrics.rmse', 'metrics.r2']].sort_values('start_time', ascending=False).head(10)